In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

import commons, models, utils, losses
from cough_datasets import CoughDatasets, CoughDatasetsCollate, CoughDetectionRatioBatchSampler

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")
#######################################################################
parser = argparse.ArgumentParser()
parser.add_argument("--init", action="store_true")
parser.add_argument("--model_name", type=str, default="try_wavlmlora_downstream")
parser.add_argument("--config_path", type=str, default="configs/ssl_finetuning.json")
args = parser.parse_args(["--model_name", "ssl_cough_highrank"])

model_dir = os.path.join("./logs", args.model_name)
config_save_path = os.path.join(model_dir, "config.json")
with open(config_save_path, "r") as f:
    data = f.read()
config = json.loads(data)

hps = utils.HParams(**config)
hps.model_dir = model_dir
hps.data.mae_training = hps.train.mae_training
hps.data.ssccl_training = hps.train.ssccl_training

# =============================================================
# SECTION: Loading Data
# =============================================================
df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df_train = df_train[hps.data.column_order]
df_test = df_test[hps.data.column_order]

collate_fn = CoughDatasetsCollate(hps.data.many_class)
# =============================================================
# SECTION: Model Setup
# =============================================================
logger = utils.get_logger(hps.model_dir)
logger.info(hps)

logger.info(f"======================================")
logger.info(f"✨ Loss: {hps.train.loss_function}")
logger.info(f"✨ Use Between Class Training: {hps.data.mix_audio}")
logger.info(f"✨ Use Augment: {hps.data.augment_data}")
logger.info(f"✨ Use Augment: Prob {hps.data.augment_prob}")
logger.info(f"✨ Use Rawboost Augment: {hps.data.augment_rawboost}")
logger.info(f"✨ Padding Type: {hps.data.pad_types}")
logger.info(f"✨ Using Model: {hps.model.pooling_model}")
logger.info(f"======================================")

hps.model.spk_dim = 0
import sys, importlib.util, shutil, tempfile
temp_path = tempfile.NamedTemporaryFile(suffix=".py", delete=False).name
shutil.copy(f"{model_dir}/model_net.py.bak", temp_path)
spec = importlib.util.spec_from_file_location("model_net", temp_path)
model_net = importlib.util.module_from_spec(spec)
sys.modules["model_net"] = model_net
spec.loader.exec_module(model_net)

pool_net = getattr(model_net, hps.model.pooling_model)
pool_model = pool_net(hps.model.feature_dim, **hps.model)

# =============================================================
# SECTION: Loop Setup
# =============================================================

class CoughDetectionRunner(L.LightningModule):
    def __init__(self, model, hps, class_weights=[]):
        super().__init__()
        self.model = model
        self.hps = hps
        self.class_weights = class_weights
        self.prev_cough_emb = None

        ssl_model = None
        if hps.model.pooling_model.split("_")[0] == "WavLMEncoder":
            logger.info("Loaded Pretrained WavLM")
            from transformers import AutoModel
            ssl_model = AutoModel.from_pretrained("microsoft/wavlm-large")
            self.model.feature_extractor.load_state_dict(ssl_model.feature_extractor.state_dict())
            self.model.feature_extractor._freeze_parameters()
            del ssl_model
            torch.cuda.empty_cache()
        
        if hps.model.ssl_model_type.lower() == "wavlm":
            from wrapper.wavlm_plus import WavLMWrapper
            ssl_model = WavLMWrapper(hps.model)
            if ssl_model != None:
                trainable_params = sum(p.numel() for p in ssl_model.parameters() if p.requires_grad)
                total_params = sum(p.numel() for p in ssl_model.parameters())
                trainable_percentage = 100 * trainable_params / total_params if total_params > 0 else 0
                logger.info(f'Trainable params: {trainable_params} | Total params: {total_params} | Trainable%: {trainable_percentage:.2f}% | Size: {trainable_params/(1e6):.2f}M')
                hps.model.feature_dim = ssl_model.hidden_size_ssl

            ssl_model.model_pooling = self.model
            self.model = ssl_model
            self.model.backbone_model.feature_extractor._freeze_parameters()


    def forward(self, x, attention_mask=None):
        x = self.model(x, attention_mask=attention_mask)
        return x
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hps.train.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val/loss",
            },
        }

    def on_after_backward(self):
        norm_type = 2
        total_norm = 0
        parameters = list(filter(lambda p: p.grad is not None, self.model.parameters()))
        for p in parameters:
            param_norm = p.grad.data.norm(norm_type)
            total_norm += param_norm.item() ** norm_type
        total_norm = total_norm ** (1. / norm_type)
        self.log("train/grad_norm", total_norm, sync_dist=True)

    def training_step(self, batch, batch_idx):
        _, audio, _, attention_masks, dse_ids, _ = batch
        
        out_model = self.forward(audio, attention_mask=attention_masks)
        tot_loss = []
        if "disease_logits" in out_model:
            ld = utils.many_loss_category(out_model["disease_logits"], dse_ids, loss_type=self.hps.train.loss_function, weights=self.class_weights)
            tot_loss.append(ld[0])

        if "loss" in out_model:
            tot_loss.append(out_model["loss"])

        # if "embedding" in out_model:
        #     tot_loss.append(self.supcon_loss(out_model["embedding"], dse_ids) * 0.03)
        #     tot_loss.append(self.center_loss(out_model["embedding"], dse_ids) * 0.005)

        loss = sum(tot_loss)
        for idx_loss, now_loss in enumerate(tot_loss):
            self.log(f"train/loss_{idx_loss}", now_loss, on_step=True, on_epoch=False, prog_bar=False, logger=True)

        self.log("train/loss_step", loss, on_step=True, on_epoch=False, prog_bar=True, logger=True)
        self.logger.experiment.add_scalars('loss', {'train': loss}, self.global_step)
        return loss

    def on_train_batch_end(self, outputs, batch, batch_idx):
        lr = self.optimizers().param_groups[0]["lr"]
        self.log("train/lr", lr, sync_dist=True)

    def on_validation_epoch_start(self):
        self.validation_step_outputs = []

    def validation_step(self, batch, batch_idx):
        _, audio, _, attention_masks, dse_ids, othr_ids = batch
        out_model = self.forward(audio, attention_mask=attention_masks)
        tot_loss = []
        if "disease_logits" in out_model:
            ld = utils.many_loss_category(out_model["disease_logits"], dse_ids, loss_type=self.hps.train.loss_function, weights=self.class_weights)
            tot_loss.append(ld[0])
        if "loss" in out_model:
            tot_loss.append(out_model["loss"])
        loss = sum(tot_loss)

        self.logger.experiment.add_scalars('loss', {'valid': loss}, self.global_step)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=False, logger=True)

        # ---- store embeddings for Phase-1 geometry ----
        emb = out_model["embedding"].detach()
        labels = dse_ids.squeeze(-1).detach()
        self.validation_step_outputs.append({
            "emb": emb,
            "labels": labels
        })

    def on_validation_epoch_end(self):
        # ---- collect embeddings ----
        emb = torch.cat([x["emb"] for x in self.validation_step_outputs], dim=0)
        labels = torch.cat([x["labels"] for x in self.validation_step_outputs], dim=0)

        cough_emb = emb[labels == 1]
        bg_emb = emb[labels == 0]

        # Guardrail: skip if batch unlucky
        if cough_emb.size(0) < 5 or bg_emb.size(0) < 5:
            return

        compactness = torch.mean(torch.cdist(cough_emb, cough_emb)).item()
        intra = torch.mean(torch.cdist(cough_emb, cough_emb))
        inter = torch.mean(torch.cdist(cough_emb, bg_emb))
        margin = (inter / intra).item()
        drift = 0.0
        if self.prev_cough_emb is not None:
            drift = torch.norm(self.prev_cough_emb.mean(0) - cough_emb.mean(0)).item()
        self.prev_cough_emb = cough_emb.detach().clone()

        # ---- log geometry ----
        self.log("val/compactness", compactness, prog_bar=False)
        self.log("val/margin", margin, prog_bar=False)
        self.log("val/drift", drift, prog_bar=False)
        self.log("val/total_geometri", (compactness * 0.25) + (2 - margin) + drift, prog_bar=False)

        # # TensorBoard (optional)
        # self.logger.experiment.add_scalars(
        #     "phase1_geometry",
        #     metrics,
        #     self.current_epoch
        # )

        # # ---- auto-stop signal ----
        # if self.phase1_monitor.should_stop():
        #     self.prev_stop_signal = True

    def on_test_epoch_start(self):
        self.test_preds = []
        self.test_labels = []

    def test_step(self, batch, batch_idx):
        _, audio, _, attention_masks, dse_ids, _ = batch

        audio = audio.float().squeeze(1)
        attention_masks = attention_masks.float()
        dse_ids = dse_ids.float()

        logits = self(audio, attention_mask=attention_masks)["disease_logits"]
        preds = torch.argmax(logits, dim=1)
        labels = torch.argmax(dse_ids, dim=1)

        self.test_preds.append(preds.cpu())
        self.test_labels.append(labels.cpu())

    def on_test_epoch_end(self):
        preds = torch.cat(self.test_preds).numpy()
        labels = torch.cat(self.test_labels).numpy()

        cm = confusion_matrix(labels, preds)
        n_classes = cm.shape[0]
        class_labels = [f"Class {i+1}" for i in range(n_classes)]

        acc = accuracy_score(labels, preds)
        b_acc = balanced_accuracy_score(labels, preds)

        sens = np.mean([
            cm[i, i] / cm[i, :].sum()
            for i in range(n_classes) if cm[i, :].sum() > 0
        ])

        spec = np.mean([
            (cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]) /
            (cm.sum() - cm[i, :].sum())
            for i in range(n_classes) if (cm.sum() - cm[i, :].sum()) > 0
        ])

        # Log metrics
        self.log("test_acc", acc, sync_dist=True)
        self.log("test_bacc", b_acc, sync_dist=True)
        self.log("test_sens", sens, sync_dist=True)
        self.log("test_spec", spec, sync_dist=True)

        # Export confusion matrix if needed
        # plt.figure(figsize=(6, 5))
        # sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
        #             xticklabels=class_labels, yticklabels=class_labels)
        # plt.xlabel("Predicted")
        # plt.ylabel("Actual")
        # plt.title("Confusion Matrix")
        # plt.savefig(f"{self.hparams.model_dir}/result_cm.png")
        # plt.close()

        return {
            "acc": acc,
            "bacc": b_acc,
            "sens": sens,
            "spec": spec,
        }

In [ ]:
runner_lightning = CoughDetectionRunner(pool_model, hps=hps, class_weights=[])
runner_lightning = CoughDetectionRunner.load_from_checkpoint(
    os.path.join("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/logs/ssl_cough_highrank/fold_0/pool_fold0_epoch=27.ckpt"),
    model=pool_model,
    hps=hps
)
runner_lightning.eval()

fold = 0
train_dataset = CoughDatasets(df_train.values, hps.data, wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{fold}.pickle", train=False)
train_loader = DataLoader(train_dataset, num_workers=28, shuffle=True, batch_size=hps.train.batch_size, pin_memory=True, drop_last=True, collate_fn=collate_fn)

test_dataset = CoughDatasets(df_test.values, hps.data, wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{fold}.pickle", train=False)
test_loader = DataLoader(test_dataset, num_workers=28, shuffle=True, batch_size=hps.train.batch_size, pin_memory=True, drop_last=True, collate_fn=collate_fn)

In [ ]:
all_embeds = []
all_labels = []

with torch.no_grad():
    for idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
        _, audio, _, attention_masks, dse_ids, _ = batch
        audio = audio.cuda()
        attention_masks = attention_masks.cuda()
        out_model = runner_lightning.forward(audio, attention_mask=attention_masks)
        embed = out_model['embedding']

        all_embeds.append(embed.cpu())
        all_labels.append(dse_ids.squeeze(-1).cpu())

all_embeds = torch.cat(all_embeds, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)
emb_2d = reducer.fit_transform(all_embeds)

In [ ]:
import matplotlib.pyplot as plt

mask_0 = all_labels == 0
mask_1 = all_labels == 1

plt.figure(figsize=(7, 6))
plt.scatter(emb_2d[mask_0, 0], emb_2d[mask_0, 1], s=10, alpha=0.6, label="Label 0")
plt.scatter(emb_2d[mask_1, 0], emb_2d[mask_1, 1], s=10, alpha=0.6, label="Label 1")
plt.legend()
plt.title("Phase-1 Embedding Space (Label 0 vs 1)")
plt.xlabel("Dim-1")
plt.ylabel("Dim-2")
plt.tight_layout()
plt.show()


In [ ]:
df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.train')
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.test')

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

#df_train = df_train[df_train['db'] == 0]
# df_train.loc[df_train['db'].isin([0, 2]), 'disease_status'] = 0
# df_train.loc[df_train['db'].isin([1, 3]), 'disease_status'] = 1

df_train = df_train[['path_file', 'disease_status', "sex", "participant"]]
df_test = df_test[['path_file', 'disease_status', "sex", "participant"]] #[['path_file', 'disease_status', "sex", "participant"]]

hps.data.cough_detection = False

fold = 0
train_dataset = CoughDatasets(df_train.values, hps.data, wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{fold}.pickle", train=False)
val_dataset = CoughDatasets(df_test.values, hps.data, wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{fold}.pickle", train=False)

train_loader = DataLoader(train_dataset, num_workers=28, shuffle=True, batch_size=hps.train.batch_size,
                            pin_memory=True, drop_last=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, num_workers=28, shuffle=False, batch_size=hps.train.batch_size,
                        pin_memory=True, drop_last=True, collate_fn=collate_fn)

In [ ]:
all_embeds = []
all_labels = []

with torch.no_grad():
    for idx, batch in tqdm(enumerate(val_loader), total=len(val_loader)):
        _, audio, _, attention_masks, dse_ids, _ = batch
        audio = audio.cuda()
        attention_masks = attention_masks.cuda()
        out_model = runner_lightning.forward(audio, attention_mask=attention_masks)
        embed = out_model['embedding']

        all_embeds.append(embed.cpu())
        all_labels.append(dse_ids.squeeze(-1).cpu())

all_embeds = torch.cat(all_embeds, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

In [ ]:
from cleanlab.filter import find_label_issues
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict

# Get predicted probabilities through cross validation.
model = XGBClassifier(tree_method="hist", enable_categorical=True)
pred_probs = cross_val_predict(model, all_embeds, all_labels, method='predict_proba')

In [ ]:
pred_probs.shape

In [ ]:
# Returns list of indices of label issues, sorted by self_confidence.
issue_idx = find_label_issues(all_labels.astype(np.int32).tolist(), pred_probs, return_indices_ranked_by='self_confidence')

# # Filter original data to show students with grade issues.
# issue_stud_id = df_train.iloc[issue n_idx].stud_ID.values.tolist()
# issues_df = df_c[df_c['stud_ID'].isin(issue_stud_id)]
# issues_df.sort_values(by="stud_ID", key=lambda column: column.map(lambda e: issue_stud_id.index(e)), inplace=True)

# # Show a few good examples.
# issues_df.head()

In [ ]:
len(df_train.iloc[issue_idx])

In [ ]:
issue_idx

In [ ]:
from sklearn.model_selection import cross_val_predict

cv_pred_probs = cross_val_predict(estimator=model, X=all_embeds, y=all_labels, cv=10, method="predict_proba")


In [ ]:
cv_accuracy = (cv_pred_probs.argmax(axis=1) == all_labels).mean()
print(f"Cross-validated estimate of accuracy on held-out data: {cv_accuracy}")

In [ ]:
all_labels

In [ ]:
from cleanlab.filter import find_label_issues
issue_idx = find_label_issues(all_labels.astype(np.int32), cv_pred_probs, return_indices_ranked_by='self_confidence')

# Show some examples!
df_train.iloc[issue_idx].head()

In [ ]:
issue_idx

In [ ]:
X = all_embeds      # shape [N, D]
y = all_labels      # TB label (0/1), weak and noisy

from sklearn.decomposition import PCA

pca = PCA(n_components=min(50, X.shape[1]))
Z = pca.fit_transform(X)

# Remove top PCs (almost always nuisance)
Z_clean = Z[:, 2:]   # drop PC1, PC2

In [ ]:
from pydiffmap import diffusion_map as dm

dmap = dm.DiffusionMap.from_sklearn(
    n_evecs=6,
    alpha=0.5,
    epsilon='bgh'
)

E = dmap.fit_transform(Z_clean)   # shape [N, K]


In [ ]:
import numpy as np

scores = []
for k in range(E.shape[1]):
    sep = np.mean(E[y == 1, k]) - np.mean(E[y == 0, k])
    scores.append(abs(sep))

tb_axis_idx = np.argmax(scores)
tb_axis = E[:, tb_axis_idx]


In [ ]:
tb_score = (tb_axis - tb_axis.min()) / (tb_axis.max() - tb_axis.min())


In [ ]:
tb_score[y == 1].mean() > tb_score[y == 0].mean()


In [ ]:
import numpy as np

q = 0.75
tb_q = np.quantile(tb_score[y == 1], q)
non_tb_q = np.quantile(tb_score[y == 0], q)

print("TB q75:", tb_q)
print("Non-TB q75:", non_tb_q)


In [ ]:
threshold = np.quantile(tb_score, 0.85)

tb_high = np.mean(y[tb_score >= threshold])
tb_low = np.mean(y[tb_score < threshold])

print("TB rate high score:", tb_high)
print("TB rate low score:", tb_low)


In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.3,
    metric="euclidean",
    random_state=42
)

X_umap = reducer.fit_transform(all_embeds)

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=tb_score,
    cmap="viridis",
    s=6,
    alpha=0.8
)
plt.colorbar(sc, label="Latent TB severity score")
plt.title("Cough embedding space colored by latent TB axis")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()


In [ ]:
qs = np.linspace(0.1, 0.9, 9)
tb_rates = []

for q in qs:
    thr = np.quantile(tb_score, q)
    sel = tb_score >= thr
    tb_rates.append((y[sel] == 1).mean())

plt.figure(figsize=(6,4))
plt.plot(qs, tb_rates, marker="o")
plt.axhline(y.mean(), linestyle="--", label="Global TB rate")
plt.xlabel("Score quantile threshold")
plt.ylabel("TB rate above threshold")
plt.legend()
plt.title("TB enrichment vs latent score rank")
plt.tight_layout()
plt.show()


In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.3,
    n_components=2,
    metric="cosine",
    random_state=42,
)

umap_emb = reducer.fit_transform(all_embeds)  # shape [N, 2]
assert umap_emb.ndim == 2 and umap_emb.shape[1] == 2
top_q = 0.8
thr = np.quantile(tb_score, top_q)
is_high = tb_score >= thr

plt.figure(figsize=(6,6))

# background
plt.scatter(
    umap_emb[:, 0],
    umap_emb[:, 1],
    c="lightgray",
    s=5,
    alpha=0.4,
)

# highlighted region
plt.scatter(
    umap_emb[is_high, 0],
    umap_emb[is_high, 1],
    c=y[is_high],
    cmap="coolwarm",
    s=8,
    alpha=0.9,
)

plt.title(f"Top {(1-top_q)*100:.0f}% latent TB score region")
plt.tight_layout()
plt.show()


In [ ]:
bins = np.linspace(tb_score.min(), tb_score.max(), 20)
bin_ids = np.digitize(tb_score, bins)

tb_rate = []
centers = []

for b in range(1, len(bins)):
    idx = bin_ids == b
    if idx.sum() > 50:
        tb_rate.append((y[idx] == 1).mean())
        centers.append(tb_score[idx].mean())

plt.figure(figsize=(6,4))
plt.plot(centers, tb_rate, marker="o")
plt.xlabel("Latent TB score")
plt.ylabel("Empirical TB probability")
plt.title("Latent TB axis calibration")
plt.tight_layout()
plt.show()


In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)
emb_2d = reducer.fit_transform(all_embeds)

import matplotlib.pyplot as plt

mask_0 = all_labels == 0
mask_1 = all_labels == 1

plt.figure(figsize=(7, 6))
plt.scatter(emb_2d[mask_0, 0], emb_2d[mask_0, 1], s=10, alpha=0.6, label="Label 0")
plt.scatter(emb_2d[mask_1, 0], emb_2d[mask_1, 1], s=10, alpha=0.6, label="Label 1")
plt.legend()
plt.title("Phase-1 Embedding Space (Label 0 vs 1)")
plt.xlabel("Dim-1")
plt.ylabel("Dim-2")
plt.tight_layout()
plt.show()
